<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Appendix A: Introduction to PyTorch (Part 1)

## A.1 What is PyTorch

In [160]:
import torch

print(torch.__version__)

2.11.0+cu128


I spent half a day configuring the PyTorch environment dependencies. I have to say, CUDA is a real pain to get right! I’ll be documenting my installation process in detail in my README.

In [161]:
print(torch.cuda.is_available())

True


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/1.webp" width="400px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/2.webp" width="300px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/3.webp" width="300px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/4.webp" width="500px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/5.webp" width="500px">

## A.2 Understanding tensors

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/6.webp" width="400px">

### A.2.1 Scalars, vectors, matrices, and tensors

In [162]:
import torch
import numpy as np

# create a 0D tensor (scalar) from a Python integer
tensor0d = torch.tensor(1)

# create a 1D tensor (vector) from a Python list
tensor1d = torch.tensor([1, 2, 3])

# create a 2D tensor from a nested Python list
tensor2d = torch.tensor([[1, 2], 
                         [3, 4]])

# create a 3D tensor from a nested Python list
tensor3d_1 = torch.tensor([[[1, 2], [3, 4]], 
                           [[5, 6], [7, 8]]])

# create a 3D tensor from NumPy array
ary3d = np.array([[[1, 2], [3, 4]], 
                  [[5, 6], [7, 8]]])
tensor3d_2 = torch.tensor(ary3d)  # Copies NumPy array
tensor3d_3 = torch.from_numpy(ary3d)  # Shares memory with NumPy array

In [163]:
ary3d[0, 0, 0] = 999
print(tensor3d_2) # remains unchanged

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])


In [164]:
print(tensor3d_3) # changes because of memory sharing

tensor([[[999,   2],
         [  3,   4]],

        [[  5,   6],
         [  7,   8]]])


### A.2.2 Tensor data types

In [165]:
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)

torch.int64


In [166]:
floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

torch.float32


In [167]:
floatvec = tensor1d.to(torch.float32)
print(floatvec.dtype)

torch.float32


### A.2.3 Common PyTorch tensor operations

In [168]:
tensor2d = torch.tensor([[1, 2, 3], 
                         [4, 5, 6]])
tensor2d

tensor([[1, 2, 3],
        [4, 5, 6]])

In [169]:
tensor2d.shape

torch.Size([2, 3])

In [170]:
tensor2d.reshape(3, 2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [171]:
tensor2d.view(3, 2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [172]:
tensor2d.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [173]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [174]:
tensor2d @ tensor2d.T

tensor([[14, 32],
        [32, 77]])

I have learned a series of formulas for tensor computation, including: definition, retrieving shapes, reshaping, transposition, and multiplication.

## A.3 Seeing models as computation graphs

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/7.webp" width="600px">

In [175]:
import torch.nn.functional as F

y = torch.tensor([1.0])  # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # weight parameter
b = torch.tensor([0.0])  # bias unit

z = x1 * w1 + b          # net input
a = torch.sigmoid(z)     # activation & output

loss = F.binary_cross_entropy(a, y)
print(loss)

tensor(0.0852)


## A.4 Automatic differentiation made easy

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/8.webp" width="600px">

In [176]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b 
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


About "retain_graph":Keep this computational graph in memory for subsequent calculation calls.

"Backpropagation" proceeds from the output layer (loss) backward through the network to the input layer, calculating the gradients of the loss with respect to each parameter (weights and biases).

In [177]:
loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


Tensor will have a .grad attribute after the .backward() function is called.(Prerequisites: requires_grad=True is enabled and the tensor is a Leaf Tensor.)

## A.5 Implementing multilayer neural networks

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/9.webp" width="500px">

In [178]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
                
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

Real feature extraction happens in the hidden layers:

50 -> 30 -> 20 features are progressively extracted.

The last layer is no longer feature extraction — it's a classifier

that takes the final 20 features and produces 3 class scores (logits).

In [179]:
model = NeuralNetwork(50, 3)

In [180]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [181]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad) #if p.requires_grad ensures that we only count parameters that will be updated during training, excluding any frozen parameters.
#p.numel() returns the total number of elements in the parameter tensor p, and the sum aggregates this count across all trainable parameters in the model.
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


I add a piece of code for viewing the dimension. Also, I add the code for viewing the bias and its dimension.

In [182]:
print(model.layers[0].weight)
print(model.layers[0].weight.shape)
print(model.layers[0].bias)
print(model.layers[0].bias.shape)

Parameter containing:
tensor([[ 0.1388,  0.0159,  0.1215,  ...,  0.1032,  0.0296,  0.0102],
        [ 0.0229,  0.0260, -0.0458,  ..., -0.0358,  0.0362,  0.0497],
        [-0.0896,  0.0113,  0.1370,  ...,  0.1037,  0.1230, -0.0929],
        ...,
        [-0.1362, -0.0713, -0.0010,  ...,  0.1176,  0.1054, -0.1012],
        [ 0.1226,  0.0937, -0.1409,  ...,  0.1321, -0.0613,  0.0086],
        [-0.0045, -0.0604,  0.0535,  ...,  0.0697,  0.0373,  0.0923]],
       requires_grad=True)
torch.Size([30, 50])
Parameter containing:
tensor([-0.0181,  0.1404,  0.0374,  0.1102,  0.0045,  0.0788,  0.1013,  0.0211,
         0.1191, -0.1204, -0.0152, -0.0222, -0.0056,  0.0466, -0.0365, -0.0321,
         0.0927, -0.1029, -0.0093,  0.1047,  0.1279, -0.1176,  0.0445,  0.0583,
         0.0263,  0.0459, -0.0549,  0.0258,  0.0305,  0.0463],
       requires_grad=True)
torch.Size([30])


In [183]:
# Set the random seed to 123 to ensure consistent results across runs (reproducibility).
torch.manual_seed(123)

# Create a neural network instance with an input layer of 50 neurons and an output layer of 3 neurons.
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [184]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [185]:
torch.manual_seed(123)

X = torch.rand((1, 50))  # Create a random input tensor of shape (1, 50), representing 50 features for a single sample. 
out = model(X)  # Condense / map the 50-dimensional information into 3-dimensional information
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


`grad_fn=<AddmmBackward0>` indicates this tensor was produced by a fully connected layer (matrix multiplication + bias addition)

 The rand function generates random numbers between 0 and 1, and (1, 50) indicates generating a 2D tensor with 1 row and 50 columns, i.e., 50 features for one sample.

In [186]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


`-0.1262,  0.1080, -0.1792` are logits. It indicates the model currently slightly leans toward classifying this feature as the second category (class 2)

In [187]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])


Softmax converts raw scores (logits) into probabilities that sum to 1

In PyTorch, dim=0 represents rows (batch size / number of samples), dim=1 represents columns (number of classes)

## A.6 Setting up efficient data loaders

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/10.webp" width="600px">

In [188]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

Note: PyTorch requires class labels to start from 0, and the maximum label must not exceed (num_classes - 1).

In [189]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])

X_test represents the test inputs, and y_test represents the test labels.

In [190]:
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]        
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [191]:
print(len(train_ds))
print(len(test_ds))
print(test_ds[0])
print(train_ds[4])

5
2
(tensor([-0.8000,  2.8000]), tensor(0))
(tensor([ 2.7000, -1.5000]), tensor(1))


I called another function to print the test data, and I found it very interesting: there is a special syntax requirement when calling `getitem` — you can only use statements like `test_ds[]`.

In [192]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2, # how many samples per batch to load. In this case, we will load 2 samples at a time during training.
    shuffle=True, # whether to shuffle the data at the beginning of each epoch (recommended for training to improve generalization)
    num_workers=0 # number of subprocesses to use for data loading. 0 means that the data will be loaded in the main process (no parallelism).
)

In [193]:
test_ds = ToyDataset(X_test, y_test)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False, # for testing, we typically do not shuffle the data to maintain a consistent order for evaluation
    num_workers=0
)

In [194]:
for idx, (x, y) in enumerate(train_loader): # enumerate() allows us to loop over something and have an automatic counter
    print(f"Batch {idx+1}:", x, y) # print the batch index and the data, `f` allows us to format the string with variables

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [195]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

  `drop_last=True`The last batch of each training epoch will be discarded to reduce its impact on model convergence during training.

In [196]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/appendix-a_compressed/11.webp" width="600px">

Generally, we set `num_workers` greater than 0 to enable data loading on a separate thread, which improves training efficiency.However, since our dataset is relatively small and we are running in *Jupyter Notebook*, we keep it set to 0 to avoid potential errors.

## A.7 A typical training loop

In [197]:
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):
    
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)
        
        loss = F.cross_entropy(logits, labels) # Loss function
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 002/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 002/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 002/002 | Train/Val Loss: 0.00


In [198]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)

print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [199]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])
tensor([0, 0, 0, 1, 1])


In [200]:
predictions = torch.argmax(outputs, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [201]:
predictions == y_train

tensor([True, True, True, True, True])

In [202]:
torch.sum(predictions == y_train)

tensor(5)

In [203]:
def compute_accuracy(model, dataloader):

    model.eval()
    correct = 0.0
    total_examples = 0
    
    for idx, (features, labels) in enumerate(dataloader):
        
        with torch.no_grad():
            logits = model(features)
        
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [204]:
compute_accuracy(model, train_loader)

1.0

In [205]:
compute_accuracy(model, test_loader)

1.0

## A.8 Saving and loading models

In [206]:
torch.save(model.state_dict(), "model.pth")

In [207]:
model = NeuralNetwork(2, 2) # needs to match the original model exactly
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

## A.9 Optimizing training performance with GPUs

### A.9.1 PyTorch computations on GPU devices

See [code-part2.ipynb](code-part2.ipynb)

### A.9.2 Single-GPU training

See [code-part2.ipynb](code-part2.ipynb)

### A.9.3 Training with multiple GPUs

See [DDP-script.py](DDP-script.py)